## 1. Setup Mountainsort
Before this notebook, need to do 
- (1) Clusterless sorters Notebook 5 in the General Dataprocessing first. That notebook detects lick artifact time.
- (2) decodePrep in Notebook 6. This step unions artifact times detected from the left and right hesmisphere and propagate to every tetrode so that every tetrode has the same artifact times. In addition, this step intersects with the track time obtained from Statescript.)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
#import cupy as cp
import numpy as np
import datajoint as dj
import spyglass as nd
import pandas as pd
import matplotlib.pyplot as plt
import json
import multiprocessing

# ignore datajoint+jupyter async warnings
import warnings
warnings.simplefilter('ignore', category=DeprecationWarning)
warnings.simplefilter('ignore', category=ResourceWarning)

from spyglass.common import (Session, IntervalList,LabMember, LabTeam, Raw, Session, Nwbfile,
                            Electrode,LFPBand,interval_list_intersect, RawPosition, TaskEpoch)
import spyglass.spikesorting as ss

[2025-07-13 19:16:44,586][INFO]: DataJoint 0.14.4 connected to shijiegu-alt@lmf-db.cin.ucsf.edu:3306


In [3]:
from spyglass.spikesorting.v0 import (SortGroup, 
                                    SortInterval,
                                    SpikeSortingPreprocessingParameters,
                                    SpikeSortingRecording, 
                                    SpikeSorterParameters,
                                    SpikeSortingRecordingSelection,
                                    ArtifactDetectionParameters, ArtifactDetectionSelection,
                                    ArtifactRemovedIntervalList, ArtifactDetection,
                                      SpikeSortingSelection, SpikeSorting,
                                   CuratedSpikeSortingSelection,CuratedSpikeSorting,Curation)
from spyglass.spikesorting.v0.curation_figurl import CurationFigurl,CurationFigurlSelection
from spyglass.spikesorting.v0.spikesorting_curation import MetricParameters,MetricSelection,QualityMetrics
from spyglass.spikesorting.v0.spikesorting_curation import WaveformParameters,WaveformSelection,Waveforms
from spyglass.utils.nwb_helper_fn import get_nwb_copy_filename
from pprint import pprint

from spyglass.shijiegu.helpers import interval_union
from spyglass.shijiegu.Analysis_SGU import TrialChoice,RippleTimes,EpochPos
from spyglass.shijiegu.load import load_run_sessions
from spyglass.shijiegu.singleUnit import do_mountainSort
from spyglass.shijiegu.decodeHelpers import runSessionNames,sleepSessionNames

### 0. Double check parameter

In [4]:
sorter_params_name = "CA1_tet_Shijie_whiten"
sorter_params = (SpikeSorterParameters & {"sorter_params_name": sorter_params_name}).fetch1("sorter_params")
sorter_params

{'detect_sign': -1,
 'adjacency_radius': -1,
 'freq_min': 0,
 'freq_max': 0,
 'filter': False,
 'whiten': True,
 'num_workers': 4,
 'clip_size': 39,
 'detect_threshold': 3,
 'detect_interval': 10,
 'verbose': True}

### 1. specify data

In [30]:
nwb_copy_file_name = "klein20231101_.nwb"

In [31]:
SortInterval & {'nwb_file_name': nwb_copy_file_name}

nwb_file_name name of the NWB file,sort_interval_name name for this interval,sort_interval 1D numpy array with start and end time for a single interval to be used for spike sorting
klein20231101_.nwb,01_Rev2Sleep1,=BLOB=
klein20231101_.nwb,02_Rev2Session1,=BLOB=
klein20231101_.nwb,03_Rev2Sleep2,=BLOB=
klein20231101_.nwb,04_Rev2Session2,=BLOB=
klein20231101_.nwb,05_Rev2Sleep3,=BLOB=
klein20231101_.nwb,06_Rev2Session3,=BLOB=
klein20231101_.nwb,07_Rev2Sleep4,=BLOB=
klein20231101_.nwb,08_Rev2Session4,=BLOB=
klein20231101_.nwb,09_Rev2Sleep5,=BLOB=
klein20231101_.nwb,10_Rev2Session5,=BLOB=


In [32]:
intervals, _ = runSessionNames(nwb_copy_file_name)
print(intervals)

['02_Rev2Session1', '04_Rev2Session2', '06_Rev2Session3', '08_Rev2Session4', '10_Rev2Session5', '12_Rev2Session6']


In [33]:
intervals2, _ = sleepSessionNames(nwb_copy_file_name)
print(intervals2)

['01_Rev2Sleep1', '03_Rev2Sleep2', '05_Rev2Sleep3', '07_Rev2Sleep4', '09_Rev2Sleep5', '11_Rev2Sleep6', '13_Rev2Sleep7']


### 2. Do sorting

In [34]:
tetrode_with_cell=np.unique((SpikeSortingRecordingSelection & {'nwb_file_name':nwb_copy_file_name}).fetch('sort_group_id'))

In [35]:
tetrode_with_cell

array([  0,  10,  11,  12,  15,  16,  28,  30,  36,  45,  52, 100, 101])

In [ ]:
for session_name in intervals:
    do_mountainSort(nwb_copy_file_name,session_name)

/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/spikeinterface/core/base.py:1038: UserWarning: Versions are not the same. This might lead to compatibility errors. Using spikeinterface==0.98.2 is recommended
  warnings.warn(
[19:18:10][INFO] Spyglass: Running spike sorting on {'nwb_file_name': 'lewis20240105_.nwb', 'sort_group_id': 0, 'sort_interval_name': '02_Rev2Session1', 'preproc_params_name': 'franklab_tetrode_hippocampus', 'team_name': 'SequenceTask', 'sorter': 'mountainsort4', 'sorter_params_name': 'CA1_tet_Shijie_whiten', 'artifact_removed_interval_list_name': 'lewis20240105_.nwb_02_Rev2Session1_0_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only'}...
INFO:spyglass:Running spike sorting on {'nwb_file_name': 'lewis20240105_.nwb', 'sort_group_id': 0, 'sort_interval_name': '02_Rev2Session1', 'preproc_params_name': 'franklab_tetrode_hippocampus', 'team_name': 'SequenceTask', 'sorter': 'mountainsort4', 'sorter_params_

Mountainsort4 use the OLD spikeextractors mapped with NewToOldRecording
Using temporary directory /stelmo/nwb/tmp/tmp4n7c4ekt
Using 4 workers.
Using tempdir: /stelmo/nwb/tmp/tmp4n7c4ekt/tmpj12cpkuh
Num. workers = 4
Preparing /stelmo/nwb/tmp/tmp4n7c4ekt/tmpj12cpkuh/timeseries.hdf5...
Preparing neighborhood sorters (M=4, N=79014380)...
Neighboorhood of channel 2 has 4 channels.
Detecting events on channel 3 (phase1)...
Elapsed time for detect on neighborhood: 0:00:05.461189
Num events detected on channel 3 (phase1): 166829
Computing PCA features for channel 3 (phase1)...
Clustering for channel 3 (phase1)...
Found 11 clusters for channel 3 (phase1)...
Computing templates for channel 3 (phase1)...
Re-assigning events for channel 3 (phase1)...
Re-assigning 157 events from 3 to 4 with dt=-2 (k=2)
Re-assigning 586 events from 3 to 2 with dt=-3 (k=3)
Re-assigning 4 events from 3 to 4 with dt=-2 (k=8)
Neighboorhood of channel 0 has 4 channels.
Detecting events on channel 1 (phase1)...
Elapsed t

/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/spikeinterface/sorters/basesorter.py:254: ResourceWarning: unclosed file <_io.TextIOWrapper name='/stelmo/nwb/recording/lewis20240105_.nwb_02_Rev2Session1_0_franklab_tetrode_hippocampus/traces_cached_seg0.raw' mode='r' encoding='UTF-8'>
  SorterClass._run_from_folder(sorter_output_folder, sorter_params, verbose)


mountainsort4 run time 102.13s


[19:19:55][INFO] Spyglass: Saving sorting results...
INFO:spyglass:Saving sorting results...
/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/spikeinterface/core/basesorting.py:237: UserWarning: The registered recording will not be persistent on disk, but only available in memory
  warnings.warn("The registered recording will not be persistent on disk, but only available in memory")
/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/datajoint/autopopulate.py:335: ResourceWarning: unclosed file <_io.TextIOWrapper name='/stelmo/nwb/recording/lewis20240105_.nwb_02_Rev2Session1_0_franklab_tetrode_hippocampus/traces_cached_seg0.raw' mode='r' encoding='UTF-8'>
  make(dict(key), **(make_kwargs or {}))
/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/tempfile.py:869: ResourceWarning: Implicitly cleaning up <TemporaryDirectory '/stelmo/nwb/tmp/tmp4n7c4ekt'>
  _warnings.warn(warn_message, ResourceWarning)
/home/shijiegu/anaconda3/envs/spyglass/lib/python3

...

Mountainsort4 use the OLD spikeextractors mapped with NewToOldRecording
Using temporary directory /stelmo/nwb/tmp/tmpl98kmgk5
Using 4 workers.
Using tempdir: /stelmo/nwb/tmp/tmpl98kmgk5/tmp4cff35v2
Num. workers = 4
Preparing /stelmo/nwb/tmp/tmpl98kmgk5/tmp4cff35v2/timeseries.hdf5...
Preparing neighborhood sorters (M=4, N=79014380)...
Neighboorhood of channel 1 has 4 channels.
Detecting events on channel 2 (phase1)...
Elapsed time for detect on neighborhood: 0:00:05.792854
Num events detected on channel 2 (phase1): 210645
Computing PCA features for channel 2 (phase1)...
Clustering for channel 2 (phase1)...
Found 7 clusters for channel 2 (phase1)...
Computing templates for channel 2 (phase1)...
Re-assigning events for channel 2 (phase1)...
Re-assigning 680 events from 2 to 3 with dt=0 (k=4)
Re-assigning 466 events from 2 to 4 with dt=4 (k=5)
Re-assigning 21 events from 2 to 1 with dt=4 (k=6)
Re-assigning 132 events from 2 to 1 with dt=4 (k=7)
Neighboorhood of channel 3 has 4 channels.
De

/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/spikeinterface/sorters/basesorter.py:254: ResourceWarning: unclosed file <_io.TextIOWrapper name='/stelmo/nwb/recording/lewis20240105_.nwb_02_Rev2Session1_13_franklab_tetrode_hippocampus/traces_cached_seg0.raw' mode='r' encoding='UTF-8'>
  SorterClass._run_from_folder(sorter_output_folder, sorter_params, verbose)


mountainsort4 run time 111.07s


[19:35:26][INFO] Spyglass: Saving sorting results...
INFO:spyglass:Saving sorting results...
/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/spikeinterface/core/basesorting.py:237: UserWarning: The registered recording will not be persistent on disk, but only available in memory
  warnings.warn("The registered recording will not be persistent on disk, but only available in memory")
/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/datajoint/autopopulate.py:335: ResourceWarning: unclosed file <_io.TextIOWrapper name='/stelmo/nwb/recording/lewis20240105_.nwb_02_Rev2Session1_13_franklab_tetrode_hippocampus/traces_cached_seg0.raw' mode='r' encoding='UTF-8'>
  make(dict(key), **(make_kwargs or {}))
/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/tempfile.py:869: ResourceWarning: Implicitly cleaning up <TemporaryDirectory '/stelmo/nwb/tmp/tmpl98kmgk5'>
  _warnings.warn(warn_message, ResourceWarning)
/home/shijiegu/anaconda3/envs/spyglass/lib/python

Mountainsort4 use the OLD spikeextractors mapped with NewToOldRecording
Using temporary directory /stelmo/nwb/tmp/tmpwz5qoo_y
Using 4 workers.
Using tempdir: /stelmo/nwb/tmp/tmpwz5qoo_y/tmpl_798sl8
Num. workers = 4
Preparing /stelmo/nwb/tmp/tmpwz5qoo_y/tmpl_798sl8/timeseries.hdf5...
Preparing neighborhood sorters (M=4, N=79014380)...
Neighboorhood of channel 0 has 4 channels.
Detecting events on channel 1 (phase1)...
Elapsed time for detect on neighborhood: 0:00:05.940732
Num events detected on channel 1 (phase1): 133075
Computing PCA features for channel 1 (phase1)...
Clustering for channel 1 (phase1)...
Found 7 clusters for channel 1 (phase1)...
Computing templates for channel 1 (phase1)...
Re-assigning events for channel 1 (phase1)...
Re-assigning 3 events from 1 to 2 with dt=-2 (k=3)
Neighboorhood of channel 3 has 4 channels.
Detecting events on channel 4 (phase1)...
Elapsed time for detect on neighborhood: 0:00:05.730812
Num events detected on channel 4 (phase1): 189881
Computing 

In [ ]:
for session_name in intervals2:
    do_mountainSort(nwb_copy_file_name,session_name)

[10:23:25][INFO] Spyglass: Running spike sorting on {'nwb_file_name': 'klein20231107_.nwb', 'sort_group_id': 0, 'sort_interval_name': '01_Rev2Sleep1', 'preproc_params_name': 'franklab_tetrode_hippocampus', 'team_name': 'SequenceTask', 'sorter': 'mountainsort4', 'sorter_params_name': 'CA1_tet_Shijie_whiten', 'artifact_removed_interval_list_name': 'klein20231107_.nwb_01_Rev2Sleep1_0_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only'}...
INFO:spyglass:Running spike sorting on {'nwb_file_name': 'klein20231107_.nwb', 'sort_group_id': 0, 'sort_interval_name': '01_Rev2Sleep1', 'preproc_params_name': 'franklab_tetrode_hippocampus', 'team_name': 'SequenceTask', 'sorter': 'mountainsort4', 'sorter_params_name': 'CA1_tet_Shijie_whiten', 'artifact_removed_interval_list_name': 'klein20231107_.nwb_01_Rev2Sleep1_0_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only'}...


Mountainsort4 use the OLD spikeextractors mapped with NewToOldRecording
Using temporary directory /stelmo/nwb/tmp/tmp7gr2llgu
Using 4 workers.
Using tempdir: /stelmo/nwb/tmp/tmp7gr2llgu/tmpkdnfixum
Num. workers = 4
Preparing /stelmo/nwb/tmp/tmp7gr2llgu/tmpkdnfixum/timeseries.hdf5...
Preparing neighborhood sorters (M=4, N=11135728)...
Neighboorhood of channel 1 has 4 channels.
Detecting events on channel 2 (phase1)...
Elapsed time for detect on neighborhood: 0:00:01.011304
Num events detected on channel 2 (phase1): 48296
Computing PCA features for channel 2 (phase1)...
Clustering for channel 2 (phase1)...
Found 9 clusters for channel 2 (phase1)...
Computing templates for channel 2 (phase1)...
Re-assigning events for channel 2 (phase1)...
Re-assigning 1372 events from 2 to 4 with dt=-1 (k=4)
Re-assigning 1223 events from 2 to 1 with dt=2 (k=5)
Re-assigning 4 events from 2 to 3 with dt=-1 (k=7)
Neighboorhood of channel 3 has 4 channels.
Detecting events on channel 4 (phase1)...
Elapsed ti

/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/spikeinterface/sorters/basesorter.py:254: ResourceWarning: unclosed file <_io.TextIOWrapper name='/stelmo/nwb/recording/klein20231107_.nwb_01_Rev2Sleep1_0_franklab_tetrode_hippocampus/traces_cached_seg0.raw' mode='r' encoding='UTF-8'>
  SorterClass._run_from_folder(sorter_output_folder, sorter_params, verbose)
[10:24:33][INFO] Spyglass: Saving sorting results...
INFO:spyglass:Saving sorting results...
/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/spikeinterface/core/basesorting.py:237: UserWarning: The registered recording will not be persistent on disk, but only available in memory
  warnings.warn("The registered recording will not be persistent on disk, but only available in memory")
/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/datajoint/autopopulate.py:335: ResourceWarning: unclosed file <_io.TextIOWrapper name='/stelmo/nwb/recording/klein20231107_.nwb_01_Rev2Sleep1_0_fra

Mountainsort4 use the OLD spikeextractors mapped with NewToOldRecording
Using temporary directory /stelmo/nwb/tmp/tmp4428fnmi
Using 4 workers.
Using tempdir: /stelmo/nwb/tmp/tmp4428fnmi/tmp00dh1ebn
Num. workers = 4
Preparing /stelmo/nwb/tmp/tmp4428fnmi/tmp00dh1ebn/timeseries.hdf5...
Preparing neighborhood sorters (M=3, N=11135728)...
Neighboorhood of channel 2 has 3 channels.
Detecting events on channel 3 (phase1)...
Elapsed time for detect on neighborhood: 0:00:00.921194
Num events detected on channel 3 (phase1): 18571
Computing PCA features for channel 3 (phase1)...
Clustering for channel 3 (phase1)...
Found 3 clusters for channel 3 (phase1)...
Computing templates for channel 3 (phase1)...
Re-assigning events for channel 3 (phase1)...
Neighboorhood of channel 1 has 3 channels.
Detecting events on channel 2 (phase1)...
Elapsed time for detect on neighborhood: 0:00:00.975230
Num events detected on channel 2 (phase1): 24190
Computing PCA features for channel 2 (phase1)...
Clustering for

/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/spikeinterface/sorters/basesorter.py:254: ResourceWarning: unclosed file <_io.TextIOWrapper name='/stelmo/nwb/recording/klein20231107_.nwb_01_Rev2Sleep1_10_franklab_tetrode_hippocampus/traces_cached_seg0.raw' mode='r' encoding='UTF-8'>
  SorterClass._run_from_folder(sorter_output_folder, sorter_params, verbose)
[10:24:51][INFO] Spyglass: Saving sorting results...
INFO:spyglass:Saving sorting results...
/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/spikeinterface/core/basesorting.py:237: UserWarning: The registered recording will not be persistent on disk, but only available in memory
  warnings.warn("The registered recording will not be persistent on disk, but only available in memory")
/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/datajoint/autopopulate.py:335: ResourceWarning: unclosed file <_io.TextIOWrapper name='/stelmo/nwb/recording/klein20231107_.nwb_01_Rev2Sleep1_10_f

...

Mountainsort4 use the OLD spikeextractors mapped with NewToOldRecording
Using temporary directory /stelmo/nwb/tmp/tmp4rmfrk30
Using 4 workers.
Using tempdir: /stelmo/nwb/tmp/tmp4rmfrk30/tmpv_y19uk5
Num. workers = 4
Preparing /stelmo/nwb/tmp/tmp4rmfrk30/tmpv_y19uk5/timeseries.hdf5...
Preparing neighborhood sorters (M=3, N=55427231)...
Neighboorhood of channel 2 has 3 channels.
Detecting events on channel 3 (phase1)...
Elapsed time for detect on neighborhood: 0:00:04.848179
Num events detected on channel 3 (phase1): 56253
Computing PCA features for channel 3 (phase1)...
Clustering for channel 3 (phase1)...
Found 3 clusters for channel 3 (phase1)...
Computing templates for channel 3 (phase1)...
Re-assigning events for channel 3 (phase1)...
Neighboorhood of channel 0 has 3 channels.
Detecting events on channel 1 (phase1)...
Elapsed time for detect on neighborhood: 0:00:04.509564
Num events detected on channel 1 (phase1): 78989
Computing PCA features for channel 1 (phase1)...
Clustering for

/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/spikeinterface/sorters/basesorter.py:254: ResourceWarning: unclosed file <_io.TextIOWrapper name='/stelmo/nwb/recording/klein20231107_.nwb_03_Rev2Sleep2_12_franklab_tetrode_hippocampus/traces_cached_seg0.raw' mode='r' encoding='UTF-8'>
  SorterClass._run_from_folder(sorter_output_folder, sorter_params, verbose)


mountainsort4 run time 59.22s


[10:35:38][INFO] Spyglass: Saving sorting results...
INFO:spyglass:Saving sorting results...
/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/spikeinterface/core/basesorting.py:237: UserWarning: The registered recording will not be persistent on disk, but only available in memory
  warnings.warn("The registered recording will not be persistent on disk, but only available in memory")
/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/site-packages/datajoint/autopopulate.py:335: ResourceWarning: unclosed file <_io.TextIOWrapper name='/stelmo/nwb/recording/klein20231107_.nwb_03_Rev2Sleep2_12_franklab_tetrode_hippocampus/traces_cached_seg0.raw' mode='r' encoding='UTF-8'>
  make(dict(key), **(make_kwargs or {}))
/home/shijiegu/anaconda3/envs/spyglass/lib/python3.10/tempfile.py:869: ResourceWarning: Implicitly cleaning up <TemporaryDirectory '/stelmo/nwb/tmp/tmp4rmfrk30'>
  _warnings.warn(warn_message, ResourceWarning)
[10:35:41][INFO] Spyglass: Running spike sorting on

Mountainsort4 use the OLD spikeextractors mapped with NewToOldRecording
Using temporary directory /stelmo/nwb/tmp/tmpc2daoaa4
Using 4 workers.
Using tempdir: /stelmo/nwb/tmp/tmpc2daoaa4/tmp3r_4yzr0
Num. workers = 4
Preparing /stelmo/nwb/tmp/tmpc2daoaa4/tmp3r_4yzr0/timeseries.hdf5...
Preparing neighborhood sorters (M=4, N=55427231)...
Neighboorhood of channel 2 has 4 channels.
Detecting events on channel 3 (phase1)...


In [36]:
all_tet_list = np.unique((SpikeSorting & {"nwb_file_name": nwb_copy_file_name,"sorter":"mountainsort4"}).fetch("sort_group_id"))

In [37]:
for tet in all_tet_list:
    key = {'nwb_file_name' : nwb_copy_file_name,
           "sort_group_id": tet,"sorter":"mountainsort4"}
    assert len(SpikeSortingRecordingSelection & key) == len(SpikeSortingRecording & key)
    
(SpikeSorting & {"nwb_file_name": nwb_copy_file_name,"sorter":"mountainsort4"})

nwb_file_name name of the NWB file,sort_group_id identifier for a group of electrodes,sort_interval_name name for this interval,preproc_params_name,team_name,sorter,sorter_params_name,artifact_removed_interval_list_name,sorting_path,"time_of_sort in Unix time, to the nearest second"
klein20231101_.nwb,0,02_Rev2Session1,franklab_tetrode_hippocampus,SequenceTask,mountainsort4,CA1_tet_Shijie_whiten,klein20231101_.nwb_02_Rev2Session1_0_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only,/stelmo/nwb/sorting/klein20231101_.nwb_02_Rev2Session1_0_franklab_tetrode_hippocampus_a3ceaccb_spikesorting,1752867198
klein20231101_.nwb,0,04_Rev2Session2,franklab_tetrode_hippocampus,SequenceTask,mountainsort4,CA1_tet_Shijie_whiten,klein20231101_.nwb_04_Rev2Session2_0_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only,/stelmo/nwb/sorting/klein20231101_.nwb_04_Rev2Session2_0_franklab_tetrode_hippocampus_76002187_spikesorting,1752868727
klein20231101_.nwb,0,06_Rev2Session3,franklab_tetrode_hippocampus,SequenceTask,mountainsort4,CA1_tet_Shijie_whiten,klein20231101_.nwb_06_Rev2Session3_0_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only,/stelmo/nwb/sorting/klein20231101_.nwb_06_Rev2Session3_0_franklab_tetrode_hippocampus_e3bdfb5d_spikesorting,1752870488
klein20231101_.nwb,0,08_Rev2Session4,franklab_tetrode_hippocampus,SequenceTask,mountainsort4,CA1_tet_Shijie_whiten,klein20231101_.nwb_08_Rev2Session4_0_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only,/stelmo/nwb/sorting/klein20231101_.nwb_08_Rev2Session4_0_franklab_tetrode_hippocampus_b90b16e0_spikesorting,1752872246
klein20231101_.nwb,0,10_Rev2Session5,franklab_tetrode_hippocampus,SequenceTask,mountainsort4,CA1_tet_Shijie_whiten,klein20231101_.nwb_10_Rev2Session5_0_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only,/stelmo/nwb/sorting/klein20231101_.nwb_10_Rev2Session5_0_franklab_tetrode_hippocampus_44fc6fa3_spikesorting,1752873899
klein20231101_.nwb,0,12_Rev2Session6,franklab_tetrode_hippocampus,SequenceTask,mountainsort4,CA1_tet_Shijie_whiten,klein20231101_.nwb_12_Rev2Session6_0_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only,/stelmo/nwb/sorting/klein20231101_.nwb_12_Rev2Session6_0_franklab_tetrode_hippocampus_7f53068d_spikesorting,1752875445
klein20231101_.nwb,10,02_Rev2Session1,franklab_tetrode_hippocampus,SequenceTask,mountainsort4,CA1_tet_Shijie_whiten,klein20231101_.nwb_02_Rev2Session1_10_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only,/stelmo/nwb/sorting/klein20231101_.nwb_02_Rev2Session1_10_franklab_tetrode_hippocampus_52929884_spikesorting,1752867306
klein20231101_.nwb,10,04_Rev2Session2,franklab_tetrode_hippocampus,SequenceTask,mountainsort4,CA1_tet_Shijie_whiten,klein20231101_.nwb_04_Rev2Session2_10_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only,/stelmo/nwb/sorting/klein20231101_.nwb_04_Rev2Session2_10_franklab_tetrode_hippocampus_8d9ea9d1_spikesorting,1752868854
klein20231101_.nwb,10,06_Rev2Session3,franklab_tetrode_hippocampus,SequenceTask,mountainsort4,CA1_tet_Shijie_whiten,klein20231101_.nwb_06_Rev2Session3_10_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only,/stelmo/nwb/sorting/klein20231101_.nwb_06_Rev2Session3_10_franklab_tetrode_hippocampus_c83542bf_spikesorting,1752870620
klein20231101_.nwb,10,08_Rev2Session4,franklab_tetrode_hippocampus,SequenceTask,mountainsort4,CA1_tet_Shijie_whiten,klein20231101_.nwb_08_Rev2Session4_10_franklab_tetrode_hippocampus_ampl_1500_prop_075_1ms_artifact_removed_valid_times_track_time_only,/stelmo/nwb/sorting/klein20231101_.nwb_08_Rev2Session4_10_franklab_tetrode_hippocampus_1385d85d_spikesorting,1752872369


In [16]:
(SpikeSorting & {"nwb_file_name": nwb_copy_file_name,"sorter":"mountainsort4"}).fetch("sort_interval_name")

array([], dtype=object)